In [ ]:

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES = 5

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Expected classes:", NUM_CLASSES)

In [ ]:

# Download the flower dataset and count images in each category.

import tarfile
import urllib.request

DATASET_URL = (
    "https://storage.googleapis.com/download.tensorflow.org/"
    "example_images/flower_photos.tgz"
)

DATA_DIR = Path("/content/flower_photos")

if not DATA_DIR.exists():
    archive_path = "/content/flower_photos.tgz"

    urllib.request.urlretrieve(DATASET_URL, archive_path)

    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall("/content")

    os.remove(archive_path)

class_names = sorted(
    folder.name
    for folder in DATA_DIR.iterdir()
    if folder.is_dir() and not folder.name.startswith(".")
)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

class_counts = {}

for class_name in class_names:
    class_folder = DATA_DIR / class_name

    image_count = sum(
        1
        for path in class_folder.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )

    class_counts[class_name] = image_count

dataset_summary = pd.DataFrame({
    "Flower category": list(class_counts.keys()),
    "Number of images": list(class_counts.values()),
})

total_images = sum(class_counts.values())

print("Dataset source:", DATASET_URL)
print("\nFlower categories:")
print(dataset_summary.to_string(index=False))
print(f"\nTotal images: {total_images}")
print(f"Number of classes: {len(class_names)}")

assert len(class_names) >= 3, "At least 3 classes are required."
assert total_images >= 500, "At least 500 images are required."

print("\nDataset meets the assignment's minimum requirements.")

In [ ]:

# Display sample images from each flower category.

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for index, class_name in enumerate(class_names):
    class_folder = DATA_DIR / class_name

    image_paths = [
        path for path in class_folder.iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    ]

    image_path = random.choice(image_paths)

    image = tf.keras.utils.load_img(
        image_path,
        target_size=IMAGE_SIZE
    )

    axes[index].imshow(image)
    axes[index].set_title(class_name.capitalize())
    axes[index].axis("off")

for index in range(len(class_names), len(axes)):
    axes[index].axis("off")

plt.suptitle("Sample Images from the Five Flower Categories")
plt.tight_layout()
plt.show()

In [ ]:

# Split the images into training, validation, and test sets.

image_paths = []
image_labels = []

for label, class_name in enumerate(class_names):
    class_folder = DATA_DIR / class_name

    for path in class_folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in image_extensions:
            image_paths.append(str(path))
            image_labels.append(label)

image_paths = np.array(image_paths)
image_labels = np.array(image_labels)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths,
    image_labels,
    test_size=0.30,
    random_state=SEED,
    stratify=image_labels
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

split_summary = pd.DataFrame({
    "Dataset split": ["Training", "Validation", "Test"],
    "Images": [
        len(train_paths),
        len(val_paths),
        len(test_paths)
    ]
})

print(split_summary.to_string(index=False))
print(f"\nTotal images assigned: {len(train_paths) + len(val_paths) + len(test_paths)}")
print(f"Training proportion: {len(train_paths) / total_images:.1%}")
print(f"Validation proportion: {len(val_paths) / total_images:.1%}")
print(f"Test proportion: {len(test_paths) / total_images:.1%}")

assert len(set(train_paths) & set(val_paths)) == 0
assert len(set(train_paths) & set(test_paths)) == 0
assert len(set(val_paths) & set(test_paths)) == 0

print("\nAll images are assigned to separate splits without overlap.")

In [ ]:

# Load, resize, batch, and optimise the image datasets.

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)

    return image, label


def create_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(train_paths, train_labels, training=True)
val_ds = create_dataset(val_paths, val_labels)
test_ds = create_dataset(test_paths, test_labels)

print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())

sample_images, sample_labels = next(iter(train_ds))

print("\nImage batch shape:", sample_images.shape)
print("Label batch shape:", sample_labels.shape)
print("Pixel value range:",
      float(tf.reduce_min(sample_images)),
      "to",
      float(tf.reduce_max(sample_images)))

In [ ]:
# Augment training images and preprocess all images for MobileNetV2.

preprocess_input = keras.applications.mobilenet_v2.preprocess_input

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")


def prepare_training(images, labels):
    images = data_augmentation(images, training=True)
    images = preprocess_input(images)
    return images, labels


def prepare_evaluation(images, labels):
    images = preprocess_input(images)
    return images, labels


train_ds = train_ds.map(
    prepare_training,
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_ds = val_ds.map(
    prepare_evaluation,
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

test_ds = test_ds.map(
    prepare_evaluation,
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

print("Augmentation techniques: horizontal flip, rotation, zoom, and contrast")
print("MobileNetV2 preprocessing applied to all datasets.")

sample_images, sample_labels = next(iter(train_ds))

print("Preprocessed image batch shape:", sample_images.shape)
print(
    "Preprocessed pixel range:",
    float(tf.reduce_min(sample_images)),
    "to",
    float(tf.reduce_max(sample_images))
)

In [ ]:
# Build a MobileNetV2 model with a new five-class classification head.

base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

print(
    "\nTrainable parameters:",
    sum(
        int(tf.size(variable))
        for variable in model.trainable_variables
    )
)

In [ ]:
# Configure callbacks to manage training and reduce overfitting.

def make_callbacks():
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1
        )
    ]

print("Early stopping and learning-rate reduction are configured.")

In [ ]:
# Train the classification head while keeping MobileNetV2 frozen.

EPOCHS_PHASE1 = 15

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=make_callbacks()
)

phase1_val_accuracy = max(history_phase1.history["val_accuracy"])

print(
    f"\nPhase 1 best validation accuracy: "
    f"{phase1_val_accuracy:.2%}"
)

In [ ]:
# Fine-tune the top MobileNetV2 layers using a low learning rate.

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print(
    "Trainable MobileNetV2 layers:",
    sum(1 for layer in base_model.layers if layer.trainable)
)

print(
    "Total trainable parameters:",
    sum(
        int(tf.size(variable))
        for variable in model.trainable_variables
    )
)

In [ ]:
# Fine-tune the model and record Phase 2 validation performance.

EPOCHS_PHASE2 = 10

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=make_callbacks()
)

phase2_val_accuracy = max(history_phase2.history["val_accuracy"])

print(
    f"\nPhase 2 best validation accuracy: "
    f"{phase2_val_accuracy:.2%}"
)

In [ ]:
# Compare the best validation accuracy achieved in each training phase.

phase_comparison = pd.DataFrame({
    "Training phase": [
        "Phase 1: Classification head",
        "Phase 2: Fine-tuning"
    ],
    "Best validation accuracy": [
        phase1_val_accuracy,
        phase2_val_accuracy
    ]
})

phase_comparison["Best validation accuracy"] = (
    phase_comparison["Best validation accuracy"].map(
        lambda value: f"{value:.2%}"
    )
)

print(phase_comparison.to_string(index=False))

In [ ]:
# Evaluate the fine-tuned model on the held-out test dataset.

test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print(f"\nFinal test loss: {test_loss:.4f}")
print(f"Final test accuracy: {test_accuracy:.2%}")

In [ ]:
# Evaluate per-class predictions and display the confusion matrix.

y_true = np.concatenate([
    labels.numpy()
    for _, labels in test_ds
])

y_probabilities = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_probabilities, axis=1)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES)
)

print("Classification report:\n")
print(
    classification_report(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        target_names=class_names,
        zero_division=0
    )
)

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Confusion Matrix — Flower Classification")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.tight_layout()
plt.show()

In [ ]:
# Display six test images with predicted classes and confidence scores.

num_samples = 6
sample_indices = np.arange(min(num_samples, len(test_paths)))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for ax, index in zip(axes, sample_indices):
    image_path = test_paths[index]

    image = tf.keras.utils.load_img(
        image_path,
        target_size=IMAGE_SIZE
    )

    predicted_label = int(y_pred[index])
    actual_label = int(y_true[index])
    confidence = float(y_probabilities[index][predicted_label])

    ax.imshow(image)
    ax.set_title(
        f"Actual: {class_names[actual_label].capitalize()}\n"
        f"Predicted: {class_names[predicted_label].capitalize()}\n"
        f"Confidence: {confidence:.2%}",
        color="green" if predicted_label == actual_label else "red"
    )
    ax.axis("off")

plt.suptitle("Sample Flower Predictions and Confidence Scores")
plt.tight_layout()
plt.show()